In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
import io

# Upload and load data
uploaded = files.upload()
uploaded_filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[uploaded_filename]))

# Column renaming
df = df.rename(columns={
    "Election_Date": "election_date",
    "Election_Type": "election_type",
    "Turnout_Percentage": "turnout",
    "Total_Seats": "total_seats",
    "Party_Name_English": "party",
    "Votes": "votes",
    "Vote_Percentage": "vote_share",
    "Seats_Won": "seats"
})

# Data type conversion
df["election_date"] = pd.to_datetime(df["election_date"])
df["election_year"] = df["election_date"].dt.year

numeric_cols = ["turnout", "total_seats", "votes", "vote_share", "seats"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

# Percentage conversion and calculations
df["vote_share"] = df["vote_share"] / 100
df["seat_share"] = df["seats"] / df["total_seats"]
df["country"] = "Bulgaria"
df["election_id"] = "BG_" + df["election_date"].dt.strftime("%Y%m%d")

# Select final columns
final_cols = [
    "country", "election_date", "election_year", "election_id", "party",
    "votes", "vote_share", "seats", "seat_share", "turnout"
]
df_clean = df[final_cols].copy()

# Calculate vote-seat ratio
df_clean["vote_seat_ratio"] = df_clean["seat_share"] / df_clean["vote_share"]
df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
df_clean = df_clean.sort_values("election_date")

# Extract unique elections
elections = (
    df_clean[["election_id", "election_date"]]
    .drop_duplicates()
    .sort_values("election_date")
    .reset_index(drop=True)
)

# Years since last election
elections["prev_election_date"] = elections["election_date"].shift(1)
elections["years_since_last_election"] = (
    (elections["election_date"] - elections["prev_election_date"]).dt.days / 365.25
)

# Elections in last 10 years
def calculate_elections_in_last_10_years(elections_df):
    counts = []
    for idx, current_row in elections_df.iterrows():
        current_date = current_row['election_date']
        time_diffs = (current_date - elections_df['election_date']).dt.days / 365.25
        count = ((time_diffs > 0) & (time_diffs <= 10)).sum()
        counts.append(count)
    return counts

elections["elections_in_last_10_years"] = calculate_elections_in_last_10_years(elections)

# Merge back with main dataframe
elections_for_merge = elections[["election_id", "years_since_last_election", "elections_in_last_10_years"]]
df_clean = df_clean.merge(elections_for_merge, on="election_id", how="left")

# Save final dataset
df_clean.to_csv("bulgaria_final_corrected.csv", index=False)

# Optional: Quick correlation analysis
election_avg = df_clean.groupby(['election_id', 'elections_in_last_10_years']).agg({
    'vote_seat_ratio': 'mean'
}).reset_index()
correlation = election_avg['vote_seat_ratio'].corr(election_avg['elections_in_last_10_years'])

print(f"Data saved to: bulgaria_final_corrected.csv")
print(f"Correlation (vote-seat ratio vs elections in last 10 years): {correlation:.3f}")
print(df_clean.head(20))
files.download('bulgaria_final_corrected.csv')

Saving Bulgarian Elections - Sheet1 (1).csv to Bulgarian Elections - Sheet1 (1) (14).csv
Data saved to: bulgaria_final_corrected.csv
Correlation (vote-seat ratio vs elections in last 10 years): -0.457
     country election_date  election_year  election_id  \
0   Bulgaria    1991-10-13           1991  BG_19911013   
1   Bulgaria    1991-10-13           1991  BG_19911013   
2   Bulgaria    1991-10-13           1991  BG_19911013   
3   Bulgaria    1994-12-18           1994  BG_19941218   
4   Bulgaria    1994-12-18           1994  BG_19941218   
5   Bulgaria    1994-12-18           1994  BG_19941218   
6   Bulgaria    1994-12-18           1994  BG_19941218   
7   Bulgaria    1994-12-18           1994  BG_19941218   
8   Bulgaria    1997-04-19           1997  BG_19970419   
9   Bulgaria    1997-04-19           1997  BG_19970419   
10  Bulgaria    1997-04-19           1997  BG_19970419   
11  Bulgaria    1997-04-19           1997  BG_19970419   
12  Bulgaria    1997-04-19           1997  BG

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>